In [2]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np


In [3]:
model = SentenceTransformer('all-MiniLM-L6-v2')

text = """
LangChain is a framework for building applications with LLMs.
Langchain provides modular abstractions to combine LLMs with tools like OpenAI and Pinecone.
You can create chains, agents, memory, and retrievers.
The Eiffel Tower is located in Paris.
France is a popular tourist destination."""

sentences = [ f.strip() for f in text.split('\n') if f.strip() != '' ]

embeddings = model.encode(sentences)

threshold = 0.7
chunks = []
current_chunk = [sentences[0]]

for i in range(1, len(sentences)):
    sim = cosine_similarity([embeddings[i]], [embeddings[i-1]])[0][0]
    if sim > threshold:
        current_chunk.append(sentences[i])
    else:
        chunks.append(' '.join(current_chunk))
        current_chunk = [sentences[i]]
    
chunks.append(' '.join(current_chunk))

print("Chunks:")
for i, chunk in enumerate(chunks):
    print(f"Chunk {i+1}: {chunk}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Chunks:
Chunk 1: LangChain is a framework for building applications with LLMs. Langchain provides modular abstractions to combine LLMs with tools like OpenAI and Pinecone.
Chunk 2: You can create chains, agents, memory, and retrievers.
Chunk 3: The Eiffel Tower is located in Paris.
Chunk 4: France is a popular tourist destination.


### RAG modular coding

In [6]:
!pip install sentence-transformers scikit-learn faiss-cpu

In [9]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from langchain.vectorstores import FAISS
from langchain.embeddings import OpenAIEmbeddings
from langchain.chat_models import init_chat_model


In [16]:
from langchain_core.documents import Document
from langchain_core.runnables import RunnableLambda, RunnableMap
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
import os
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")


In [19]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from langchain_core.documents import Document

class ThresholdChunker:
    def __init__(self, model_name="all-MiniLM-L6-v2", threshold=0.7):
        self.model = SentenceTransformer(model_name)
        self.threshold = threshold

    def chunk(self, text):
        sentences = [f.strip() for f in text.split('\n') if f.strip() != '']

        if not sentences:
            return []

        embeddings = self.model.encode(sentences)

        chunks = []
        current_chunk = [sentences[0]]

        for i in range(1, len(sentences)):
            sim = cosine_similarity([embeddings[i]], [embeddings[i - 1]])[0][0]

            if sim > self.threshold:
                current_chunk.append(sentences[i])
            else:
                chunks.append(' '.join(current_chunk))
                current_chunk = [sentences[i]]

        chunks.append(' '.join(current_chunk))
        return chunks

    def split_into_chunks(self, docs):
        result = []

        for doc in docs:
            for chunk in self.chunk(doc.page_content):
                result.append(
                    Document(
                        page_content=chunk,
                        metadata=doc.metadata
                    )
                )

        return result



In [20]:
sample_text = """LangChain is a framework for building applications with LLMs.
Langchain provides modular abstractions to combine LLMs with tools like OpenAI and Pinecone.
You can create chains, agents, memory, and retrievers.
The Eiffel Tower is located in Paris.
France is a popular tourist destination."""

doc = Document(page_content=sample_text, metadata={"source": "sample.txt"})
doc


Document(metadata={'source': 'sample.txt'}, page_content='LangChain is a framework for building applications with LLMs.\nLangchain provides modular abstractions to combine LLMs with tools like OpenAI and Pinecone.\nYou can create chains, agents, memory, and retrievers.\nThe Eiffel Tower is located in Paris.\nFrance is a popular tourist destination.')

In [23]:
## chunking
chunker = ThresholdChunker(threshold=0.7)
chunks = chunker.split_into_chunks([doc])
chunks

print(f"Number of chunks: {len(chunks)}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Number of chunks: 4


In [ ]:
FAISS.from_documents(chunks, OpenAIEmbeddings())

In [24]:
import os
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")
embeddings = OpenAIEmbeddings()
vectorstore = FAISS.from_documents(chunks, embeddings)
retriver = vectorstore.as_retriever()


C:\Users\raavi\AppData\Local\Temp\ipykernel_5352\3779541176.py:3: LangChainDeprecationWarning: The class `OpenAIEmbeddings` was deprecated in LangChain 0.0.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-openai package and should be used instead. To use it run `pip install -U `langchain-openai` and import as `from `langchain_openai import OpenAIEmbeddings``.
  embeddings = OpenAIEmbeddings()


In [40]:
## Prompt Template

# --- 5. Prompt Template ---
template = """Answer the question based on the following context:

{context}

Question: {question}
"""

prompt = PromptTemplate.from_template(template)
prompt

PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template='Answer the question based on the following context:\n\n{context}\n\nQuestion: {question}\n')

In [55]:
from langchain_groq import ChatGroq

llm = init_chat_model(
    model="groq:llama-3.1-8b-instant",
    temperature=0.4
)

In [56]:
print(type(vectorstore))

<class 'langchain_community.vectorstores.faiss.FAISS'>


In [60]:
## LLM
llm=init_chat_model(model="groq:llama-3.1-8b-instant",temperature=0.4)

### LCEL Chain With retrieval

rag_chain=(
    RunnableMap(
        {
        "context": lambda x: retriever.invoke(x["question"]),
        "question": lambda x: x["question"],  
        }
    )
    | prompt
    | llm
    | StrOutputParser()
)

# --- 8. Run Query ---


In [61]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

In [62]:
query = {"question": "What is LangChain used for?"}
result = rag_chain.invoke(query)

print(result)

LangChain is a framework for building applications with LLMs (Large Language Models). It provides modular abstractions to combine LLMs with tools like OpenAI and Pinecone.


In [63]:
!pip install langchain-experimental

   ---------------------------------------- 0.0/2.4 MB ? eta -:--:--
   ---------------------------------------- 2.4/2.4 MB 44.7 MB/s  0:00:00

  Attempting uninstall: langchain-text-splitters

    Found existing installation: langchain-text-splitters 0.3.8

    Uninstalling langchain-text-splitters-0.3.8:

      Successfully uninstalled langchain-text-splitters-0.3.8

   ---------------------------------------- 0/3 [langchain-text-splitters]
  Attempting uninstall: langchain-community
   ---------------------------------------- 0/3 [langchain-text-splitters]
    Found existing installation: langchain-community 0.3.24
   ---------------------------------------- 0/3 [langchain-text-splitters]
   ------------- -------------------------- 1/3 [langchain-community]
    Uninstalling langchain-community-0.3.24:
   ------------- -------------------------- 1/3 [langchain-community]
   ------------- -------------------------- 1/3 [langchain-community]
      Successfully uninstalled langchain-com

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain 0.3.25 requires langchain-core<1.0.0,>=0.3.58, but you have langchain-core 1.4.0 which is incompatible.
langchain 0.3.25 requires langchain-text-splitters<1.0.0,>=0.3.8, but you have langchain-text-splitters 1.1.2 which is incompatible.


In [68]:
from langchain_openai import OpenAIEmbeddings
from langchain_experimental.text_splitter import SemanticChunker
from langchain_openai import OpenAIEmbeddings


In [69]:
from langchain_community.document_loaders import TextLoader

In [73]:
loader = TextLoader("langchain_intro.txt")
loader.load()

embeddings = OpenAIEmbeddings()

chunker = SemanticChunker(embeddings=embeddings)

chunks = chunker.split_documents(loader.load())

for i, chunk in enumerate(chunks):
    print(f"Chunk {i+1}: {chunk.page_content}\n")   
    

Chunk 1: LangChain is a framework for building applications with LLMs. Langchain provides modular abstractions to combine LLMs with tools like OpenAI and Pinecone.

Chunk 2: You can create chains, agents, memory, and retrievers. The Eiffel Tower is located in Paris. France is a popular tourist destination.

